# paperfetch quickstart

Search for scholarly papers and download free, legal open-access PDFs.
You don't need to know how to code: the only thing you change is the text in **Step 2**.

**How to use this notebook**

- A notebook is a list of cells. Click a cell and press **Shift + Enter** to run it and move to the next one.
- Run the cells **in order, from top to bottom**.
- Grey boxes are code. You don't need to read them, just run them.
- While a cell runs it shows `[*]`; when it's done it shows a number like `[3]`.


## Step 1: Set up

This loads paperfetch. If you have a free OpenAlex API key, paste it when asked;
otherwise just press Enter. Searching works without a key, but some PDFs can only
be downloaded with one. Get a key at <https://openalex.org/settings/api>.

You can choose to have it remembered on this computer, so you're only asked once.


In [ ]:
import csv
import os
from getpass import getpass
from pathlib import Path

import requests
from dotenv import load_dotenv, set_key
from IPython.display import Markdown, display

from paperfetch.download import download_pdf
from paperfetch.openalex import search_works

load_dotenv()  # loads whatever key you have within `.env`; otherwise, you will be prompted to enter it.
if not os.environ.get("OPENALEX_API_KEY"):
    key = getpass("Paste your OpenAlex API key, or press Enter to skip: ").strip()
    if key:
        os.environ["OPENALEX_API_KEY"] = key
        answer = input("Remember this key on this computer? Type y or n: ")
        if answer.strip().lower() == "y":
            set_key(Path("../.env"), "OPENALEX_API_KEY", key)
            print("Saved. You won't be asked again on this computer.")

if os.environ.get("OPENALEX_API_KEY"):
    print("Ready. API key is set.")
else:
    print("Ready, without an API key. Search works; OpenAlex's cached PDFs won't.")

## Step 2: What are you looking for?

**Below is is the only cell you need to change.** Type your search query between the quotation
marks, then run this cell and every cell below it.


In [ ]:
SEARCH_FOR = "open access citation advantage"
HOW_MANY = 5  # a number from 1 to 100
SAVE_TO = Path("../papers")  # the "papers" folder next to the "notebooks" folder

## Step 3: Search

Each result shows where a free PDF can come from:

- **Publisher PDF**: a direct link from the publisher or a repository. Tried first, and free.
- **OpenAlex copy**: OpenAlex keeps its own copy. Used only if the publisher link fails,
  and needs an API key (about 100 per day on a free key).

If both say **no**, the paper is free to read only as a web page. Click its title to open it.


In [ ]:
papers = search_works(SEARCH_FOR, limit=HOW_MANY)


def yes_no(value):
    return "yes" if value else "no"


rows = [
    "| # | Year | Title | First author | Publisher PDF | OpenAlex copy |",
    "|---|---|---|---|---|---|",
]
for number, paper in enumerate(papers, start=1):
    title = (paper.title or "(untitled)").replace("|", "/")
    link = paper.landing_page_url or (
        f"https://doi.org/{paper.doi}" if paper.doi else None
    )
    if link:
        title = f"[{title}]({link})"
    author = paper.authors[0] if paper.authors else ""
    rows.append(
        f"| {number} | {paper.year or ''} | {title} | {author} "
        f"| {yes_no(paper.pdf_url)} | {yes_no(paper.has_cached_pdf)} |"
    )

if papers:
    display(Markdown("\n".join(rows)))
    downloadable = sum(1 for paper in papers if paper.pdf_url or paper.has_cached_pdf)
    print(f"{downloadable} of {len(papers)} papers have at least one PDF source.")
else:
    display(Markdown("**No results.** Try fewer or more general words in Step 2."))

## Step 4: Choose what to download

Leave it as `"all"`, or list the numbers from the table, for example `[1, 3, 5]`.


In [ ]:
DOWNLOAD = "all"  # or a list of numbers, e.g. [1, 3, 5]

## Step 5: Download

Papers already in the folder are skipped, so running this again is safe and free.


In [ ]:
chosen = papers if DOWNLOAD == "all" else [papers[number - 1] for number in DOWNLOAD]

saved = []
for paper in chosen:
    try:
        path = download_pdf(paper, SAVE_TO)
    except requests.HTTPError as error:
        print(f"\nOpenAlex stopped the downloads ({error}). See Troubleshooting below.")
        break
    print(f"[{'saved ' if path else 'no PDF'}]  {paper.title}")
    if path:
        saved.append((path.name, paper))

print(f"\nSaved {len(saved)} of {len(chosen)} PDFs to: {SAVE_TO.resolve()}")

## Step 6: Make a list of which file is which

PDFs are named by their OpenAlex ID (like `W3082255212.pdf`). This saves `index.csv` in
the same folder, listing each file's title, year, authors and DOI. It opens in Excel.


In [ ]:
FIELDS = ["file", "title", "year", "authors", "doi", "search"]
index_path = SAVE_TO / "index.csv"

index = {}
if index_path.exists():
    with index_path.open(newline="", encoding="utf-8-sig") as file:
        index = {row["file"]: row for row in csv.DictReader(file)}

for file_name, paper in saved:
    index[file_name] = {
        "file": file_name,
        "title": paper.title or "",
        "year": paper.year or "",
        "authors": "; ".join(paper.authors),
        "doi": paper.doi or "",
        "search": SEARCH_FOR,
    }

SAVE_TO.mkdir(parents=True, exist_ok=True)
with index_path.open("w", newline="", encoding="utf-8-sig") as file:
    writer = csv.DictWriter(file, fieldnames=FIELDS)
    writer.writeheader()
    writer.writerows(index.values())

print(f"{len(index)} papers listed in: {index_path.resolve()}")

## Troubleshooting

- **`No module named 'paperfetch'`**: the notebook was opened the wrong way.
  Close it and start it with `pixi run notebook` (see the README).
- **No results**: use fewer or more general words in Step 2.
- **Many "no PDF"**: that's normal. Some free papers only exist as web pages;
  click their title in the Step 3 table to read them. An API key adds more PDFs.
- **"OpenAlex stopped the downloads"**: the free daily budget (about 100 cached PDFs)
  is used up. Try again tomorrow, or download fewer papers.
- **Changed Step 2 but the results didn't change**: run Step 2 *and every cell below it* again.
